In [ ]:
#TODO make ingest function using redivis API

In [2]:
import os
import redivis
from dotenv import load_dotenv

load_dotenv()

username = os.getenv("REDIVIS_USERNAME")
orgname = os.getenv("REDIVIS_ORGANIZATION") # Fixed spelling

# Verify they are not None
print(f"User: {username} | Org: {orgname}")

user = redivis.user(username)
cul_dataset_list = redivis.organization(orgname).list_datasets()

User: matteo_perini | Org: cul_hcup


In [ ]:



workflow = user.workflow("nytest:jnda")
sedd_ext = workflow.table("filter_ext_suicide_output:y71g")

# Load table as a dataframe
sedd_ext_df = sedd_ext.to_pandas_dataframe(max_results=100)
#sedd_ext_df.head()

In [ ]:
import redivis
import re

def extract_hcup_data(state_full, state_abbr, db_type, years, desired_cols, filter_cols, filter_condition):
    dataset = redivis.organization("cul_hcup").dataset(f"{state_full}")
    
    # 1. Filter for target tables
    target_tables = []
    year_pattern = "|".join(years)
    for t in dataset.list_tables():
        name = t.name.upper()
        if state_abbr.upper() in name and db_type.upper() in name and "CORE" in name:
            if re.search(year_pattern, name):
                target_tables.append(t)

    # 2. Find intersecting columns
    common_columns = None
    for t in target_tables:
        cols = {v.name.upper() for v in t.list_variables()}
        common_columns = cols if common_columns is None else common_columns.intersection(cols)

    # 3. Finalize SELECT variables
    final_cols = [c for c in desired_cols if c.upper() in common_columns]
    col_string = ", ".join(final_cols)

    # 4. Generate UNION query
    sql_parts = [f"SELECT {col_string} FROM `{t.qualified_reference}`" for t in target_tables]
    union_query = "\nUNION DISTINCT\n".join(sql_parts)

    # 5. Programmatically generate WHERE clause (ensuring filter columns actually exist)
    valid_filter_cols = [c for c in filter_cols if c.upper() in common_columns]
    where_statements = [f"{col} {filter_condition}" for col in valid_filter_cols]
    where_clause = " OR \n    ".join(where_statements)

    # 6. Assemble and execute
    final_sql = f"""
    WITH stacked_data AS (
    {union_query}
    )
    SELECT * FROM stacked_data
    WHERE 
        {where_clause};
    """
    
    return redivis.query(final_sql).to_pandas_dataframe()

# --- Example Usage ---
ny_sedd_ICD9_df = extract_hcup_data(
    state_full="new_york_hcup:cnre",
    state_abbr="ny",
    db_type="sedd",
    years=[str(y) for y in range(2006, 2015)] + ["2015q1q3"],
    desired_cols=["KEY", "AGE"] + [f"ECODE{i}" for i in range(1, 10)],
    filter_cols=[f"ECODE{i}" for i in range(1, 10)],
    filter_condition="BETWEEN 'E950' AND 'E959'"
)

